# 4.5 Difference-in-differences with spillovers

This notebook implements the spillover specification:

$$ Y_{it} = \beta \cdot \text{Post}_t \cdot \text{Treated}_i + \gamma \cdot \text{Post}_t \cdot S_i + \theta_t + \eta_i + \epsilon_{it} $$

where $S_i = \sum_{j \neq i} w_{ij} \text{Treated}_j$ is lab group $i$'s exposure to
treated lab groups, with $w_{ij}$ row-normalized so $S_i \in [0,1]$.

This notebook:
    - Creates the network proximity measures $w_{ij}$
    - Runs the regressions including the spillovers and creates regression tables

Currently, only research collaboration has a complete lab group to lab group mapping with matched labgroupids. The other four proximity measures (geographical distance, lab space sharing, equipment sharing, communication) will be added when mappings are complete.

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from itertools import combinations
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
import pyfixest as pf

sys.path.append(str(Path.cwd().parents[0] / "functions"))
from make_regression_table import make_regression_table
from spillover_helpers import normalize_weights, compute_exposure

In [2]:
# Load data

# Cleaned lab group data
df = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""]  # Only treat empty strings as NaN
)

# Publications data
publications = pd.read_csv(
    config.PUBLICATON_DATA / 
    "2_Processed" / 
    "publications_matched.csv"
)

In [3]:
# Construct post variable
df["post"] = (df["survey"] == "EL").astype(int)

# Keep only labgroups that have both pre and post observations
labgroup_counts = df.groupby("labgroupid")["survey"].nunique()
labgroups_to_keep = labgroup_counts[labgroup_counts == 2].index
df = df[df["labgroupid"].isin(labgroups_to_keep)].copy()

## (1) Constructing the network proximity measures

### (a) Research collaboration

We build $S_i$ for the research-collaboration measure using co-authorship between lab groups. We do this in the following way:

(i) We restrict publications to those dated before the first lab group's BL survey.

(ii) We then construct weights. Weights are defined as: $w_{ij}$ = (publications shared with $j$) / (total publications shared with all $j' \neq i$).

(iii) We then construct $S_i$: $S_i = \sum_{j \neq i} w_{ij} \text{Treated}_j$. If all lab group $i$'s co-authored publications are with treated lab groups, $S_i = 1$; if all lab group $i$'s co-authored publications are with control lab groups, $S_i = 0$. Lab groups with no co-authorship publications get $S_i = 0$.

Note: date in the publications data is sometimes year-only. As a robustness check, we exclude these publications.

In [4]:
# (i) Restrict publications to those dated before the first lab group's BL survey

# Pre-treatment cutoff: first lab group's BL survey date
cutoff_date = pd.to_datetime(df.loc[df["survey"] == "BL", "survey_date_bl"]).min()
print(f"Pre-treatment cutoff for network construction: {cutoff_date.date()}")


# Create clean date column. "mixed" required as date mixes year only and full dates
publications["date_parsed"] = pd.to_datetime(publications["date"], errors="coerce", format="mixed")

# Restrict to publications that are pre-treatment and link 2+ lab groups
pre_treatment_pubs = publications[
    (publications["date_parsed"] < cutoff_date) & (publications["n_matched_labgroupids"] > 1)
].copy()

print(f"{len(pre_treatment_pubs)} of {len(publications)} publications are pre-treatment and link 2+ lab groups")

Pre-treatment cutoff for network construction: 2025-10-06
396 of 8014 publications are pre-treatment and link 2+ lab groups


In [5]:
# (ii) Create weights

# Build edge weights: for each pre-treatment publication matched to 2+ labs,
# add 1 to every pair of matched labgroupids
edge_counts = {}
for ids in pre_treatment_pubs["matched_labgroupids"]:
    lab_ids = sorted(set(int(x) for x in str(ids).split(";")))
    for a, b in combinations(lab_ids, 2):
        edge_counts[(a, b)] = edge_counts.get((a, b), 0) + 1

edges = pd.DataFrame(
    [(a, b, w) for (a, b), w in edge_counts.items()],
    columns=["labgroupid_a", "labgroupid_b", "n_shared_pubs"]
)
print(f"{len(edges)} lab-group pairs share at least one pre-treatment publication")

# Restrict the network's node set to lab groups that consented to data merging
consented = df["consent_data_merge"] == "Yes I consent to this data collection and merging"
consenting_labgroupids = df.loc[consented, "labgroupid"].unique()
n_excluded = df["labgroupid"].nunique() - len(consenting_labgroupids)
print(f"{n_excluded} of {df['labgroupid'].nunique()} lab groups did not consent to publication data merging and are excluded from the collaboration network")

W_collaboration = normalize_weights(
    edges, id_col_i="labgroupid_a", id_col_j="labgroupid_b",
    weight_col="n_shared_pubs", all_ids=consenting_labgroupids
)

67 lab-group pairs share at least one pre-treatment publication
28 of 109 lab groups did not consent to publication data merging and are excluded from the collaboration network


In [6]:
# (iii) Compute S_i

# Compute S_i = sum_j W_ij * D_j, where D_j is the treatment status of lab j
treated_by_lab = df.drop_duplicates("labgroupid").set_index("labgroupid")["treated"]
S_collaboration = compute_exposure(W_collaboration, treated_by_lab)

# Non-consenting labs get S_collaboration = NaN (dropped in regressions below)
df["S_collaboration"] = df["labgroupid"].map(S_collaboration)
assert df["S_collaboration"].dropna().between(0, 1).all()

# Report distribution of S_collaboration
print(df["S_collaboration"].describe())
n_zero = (S_collaboration == 0).sum()
print(f"{n_zero} of {len(S_collaboration)} consenting lab groups have zero measured collaboration exposure")

count    162.000000
mean       0.234053
std        0.381328
min        0.000000
25%        0.000000
50%        0.000000
75%        0.500000
max        1.000000
Name: S_collaboration, dtype: float64
55 of 81 consenting lab groups have zero measured collaboration exposure


**TO-DO LIST**
Add: 
    (b) Geographical distance measure construction
    (c) Equipment sharing measure construction
    (d) Space sharing measure construction
    (e) Communication measure construction

## (2) Difference-in-differences with spillovers

We estimate the PAP spec for the primary outcome (levels and log), plus a version adding the triple interaction $\text{Post}_t \cdot \text{Treated}_i \cdot S_i$, which lets the spillover effect differ for treated vs. control lab groups.

### (a) Research collaboration

Note: since some lab groups did not consent to publication data merging, these regressions run on a smaller sample.

In [7]:
n_dropped = df["S_collaboration"].isna().sum()
print(f"{n_dropped} of {len(df)} panel rows have missing S_collaboration and will be dropped from the spillover regressions")

df["log_electricity"] = np.log1p(df["annual_electricity_total"])

# Same-sample baseline: plain treated:post, no spillover terms, restricted to
# the same consenting-lab sample as the spillover specs below -- isolates how
# much treated:post shifts from the sample restriction alone vs. controlling
# for S_collaboration
df_consenting = df.dropna(subset=["S_collaboration"])

fit_levels_baseline = pf.feols(
    "annual_electricity_total ~ treated:post | labgroupid + post",
    data=df_consenting, vcov={"CRV1": "labgroupid"}
)
fit_log_baseline = pf.feols(
    "log_electricity ~ treated:post | labgroupid + post",
    data=df_consenting, vcov={"CRV1": "labgroupid"}
)

fit_levels_main = pf.feols(
    "annual_electricity_total ~ treated:post + S_collaboration:post | labgroupid + post",
    data=df, vcov={"CRV1": "labgroupid"}
)
fit_levels_triple = pf.feols(
    "annual_electricity_total ~ treated:post + S_collaboration:post + treated:S_collaboration:post | labgroupid + post",
    data=df, vcov={"CRV1": "labgroupid"}
)
fit_log_main = pf.feols(
    "log_electricity ~ treated:post + S_collaboration:post | labgroupid + post",
    data=df, vcov={"CRV1": "labgroupid"}
)
fit_log_triple = pf.feols(
    "log_electricity ~ treated:post + S_collaboration:post + treated:S_collaboration:post | labgroupid + post",
    data=df, vcov={"CRV1": "labgroupid"}
)

fit_levels_main.summary()

56 of 218 panel rows have missing S_collaboration and will be dropped from the spillover regressions


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


###

Estimation:  OLS
Dep. var.: annual_electricity_total, Fixed effects: labgroupid + post
sample: None = all
Inference:  CRV1
Observations:  162

| Coefficient          |   Estimate |   Std. Error |   t value |   Pr(>|t|) |     2.5% |   97.5% |
|:---------------------|-----------:|-------------:|----------:|-----------:|---------:|--------:|
| treated:post         |     26.420 |       96.437 |     0.274 |      0.785 | -165.495 | 218.335 |
| S_collaboration:post |   -136.077 |       77.591 |    -1.754 |      0.083 | -290.488 |  18.334 |
---
RMSE: 219.708 R2: 1.0 R2 Within: 0.015 


In [8]:
# Export to nice table
table = make_regression_table(
    fit_list = [
        fit_levels_baseline, fit_levels_main, fit_levels_triple,
        fit_log_baseline, fit_log_main, fit_log_triple,
    ],
    model_names   = ["(1)", "(2)", "(3)", "(4)", "(5)", "(6)"],
    keep_vars     = ["treated:post", "S_collaboration:post", "treated:S_collaboration:post"],
    var_labels    = {
        "treated:post": "Treated $\\times$ Post",
        "S_collaboration:post": "Exposure (collab.) $\\times$ Post",
        "treated:S_collaboration:post": "Treated $\\times$ Exposure $\\times$ Post",
    },
    fe_rows       = {
        "Research group FE": [True] * 6,
        "Time FE":            [True] * 6,
    },
    col_groups    = {"Levels": [0, 1, 2], "Log": [3, 4, 5]},
    col_subgroups = {"Baseline": [0, 3], "Main": [1, 4], "Triple interaction": [2, 5]},
    baseline_mean  = "auto",
    outcome_levels = "annual_electricity_total",
    df_levels      = df,
    decimals       = [1, 1, 1, 3, 3, 3],
    mean_decimals  = [0, 0, 0, 3, 3, 3],
    r2_type        = None,
    col1_width     = "5.5cm",
    coln_width     = "2.1cm",
)
table_path = config.OUTPUT / "5_Regression_Tables" / "spillovers_collaboration.tex"
_ = table_path.write_text(table)

**TO-DO LIST**
Plug in each measure into the same normalize_weights and compute_exposure code as above - just swap the edge list to the measure specific edge list.